In [12]:
import boto3

# # Assume into OrganizationAccountAccessRole in a target account
# # By default this uses your *current* account ID; change TARGET_ACCOUNT_ID
# # if you want to assume into a different AWS account.
base_session = boto3.session.Session()
sts = base_session.client("sts")

current_identity = sts.get_caller_identity()
print(current_identity)



print("AWS STS get_caller_identity() after assume-role:")
print(f"  Account:   {current_identity['Account']}")
print(f"  UserId:    {current_identity['UserId']}")
print(f"  ARN:       {current_identity['Arn']}")
print(f"  Region:    {base_session.region_name}")


{'UserId': 'AIDAX4GAO7NCBZDMKVFDS', 'Account': '541569514308', 'Arn': 'arn:aws:iam::541569514308:user/christiaan.vanderberg', 'ResponseMetadata': {'RequestId': 'b7511da6-6122-420a-acae-b7c3221e4fbb', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'b7511da6-6122-420a-acae-b7c3221e4fbb', 'x-amz-sts-extended-request-id': 'MTpldS13ZXN0LTE6UzoxNzY4MjQzNDUyMDYwOlI6dTRrM0lPOVU=', 'content-type': 'text/xml', 'content-length': '418', 'date': 'Mon, 12 Jan 2026 18:44:12 GMT'}, 'RetryAttempts': 0}}
AWS STS get_caller_identity() after assume-role:
  Account:   541569514308
  UserId:    AIDAX4GAO7NCBZDMKVFDS
  ARN:       arn:aws:iam::541569514308:user/christiaan.vanderberg
  Region:    eu-west-1


In [ ]:
import boto3
import json
from datetime import datetime, timezone

LOG_GROUP_NAME = "/ecs/prod-ComotionAuth"
start_day = 5
end_day = 20

start_dt = datetime(2025, 12, start_day, 0, 0, 0, tzinfo=timezone.utc)
end_dt = datetime(2025, 12, end_day, 23, 59, 59, tzinfo=timezone.utc)

start_ms = int(start_dt.timestamp() * 1000)
end_ms = int(end_dt.timestamp() * 1000)

realm = "RGASA"
output_path = f"comoauth_api_{realm}_logs_2025-12-{start_day}_to_2025-12-{end_day}.jsonl"

logs_client = boto3.client("logs")

all_events = []
next_token = None
page = 0
MAX_EVENTS = 1000  # <-- total cap

while True:
    if len(all_events) >= MAX_EVENTS:
        print(f"Reached MAX_EVENTS={MAX_EVENTS}, stopping pagination.")
        break

    params = {
        "logGroupName": LOG_GROUP_NAME,
        "startTime": start_ms,
        "endTime": end_ms,
        "limit": 10000,
    }
    if next_token is not None:
        params["nextToken"] = next_token

    response = logs_client.filter_log_events(**params)
    page += 1

    raw_events = response.get("events", [])
    events = [
        e for e in raw_events
        if f"realmId={realm}" in e.get("message", "")
        and "realmId=" in e.get("message", "")
        and "type=REFRESH_TOKEN_ERROR" not in e.get("message", "")
    ]

    all_events.extend(events)
    print(
        f"Fetched page {page}, {len(raw_events)} events, "
        f"{len(events)} matched realmId={realm} and not REFRESH_TOKEN_ERROR "
        f"(total matches so far: {len(all_events)})"
    )

    next_token = response.get("nextToken")
    if not next_token:
        break

# Trim to MAX_EVENTS just in case
all_events = all_events[:MAX_EVENTS]

with open(output_path, "w", encoding="utf-8") as f:
    for event in all_events:
        f.write(json.dumps(event, ensure_ascii=False))
        f.write("\n")

print(f"Downloaded {len(all_events)} events from '{LOG_GROUP_NAME}' between {start_dt} and {end_dt} into {output_path}.")

Fetched page 1, 2027 events, 10 matched realmId=RGASA and not REFRESH_TOKEN_ERROR (total matches so far: 10)
Fetched page 2, 1930 events, 9 matched realmId=RGASA and not REFRESH_TOKEN_ERROR (total matches so far: 19)
Fetched page 3, 2144 events, 14 matched realmId=RGASA and not REFRESH_TOKEN_ERROR (total matches so far: 33)
Fetched page 4, 2980 events, 36 matched realmId=RGASA and not REFRESH_TOKEN_ERROR (total matches so far: 69)
Fetched page 5, 2854 events, 43 matched realmId=RGASA and not REFRESH_TOKEN_ERROR (total matches so far: 112)
Fetched page 6, 782 events, 11 matched realmId=RGASA and not REFRESH_TOKEN_ERROR (total matches so far: 123)
Fetched page 7, 924 events, 12 matched realmId=RGASA and not REFRESH_TOKEN_ERROR (total matches so far: 135)
Fetched page 8, 332 events, 5 matched realmId=RGASA and not REFRESH_TOKEN_ERROR (total matches so far: 140)
Fetched page 9, 806 events, 11 matched realmId=RGASA and not REFRESH_TOKEN_ERROR (total matches so far: 151)
Fetched page 10, 122